# Práctica 2. Pipeline ETL y Limpieza de Datos con Pandas

**Materia:** Extracción de conocimientos en Base de Datos  
**Actividad:** Limpieza y transformación del archivo `pokemon.csv`  
**Alumno:** _Escribe tu nombre_  
**Grupo:** 9 IDGS  
**Maestra:** M.C. Martha Cárdenas Maciel  
**Fecha:** _Escribe la fecha de entrega_

## Objetivo
Construir un flujo ETL (Extract, Transform, Load) con Pandas para inspeccionar el conjunto de datos de Pokémon, corregir problemas de calidad y exportar un archivo limpio para su análisis.

> **Antes de comenzar:** coloca este notebook en la misma carpeta que `pokemon.csv`. Ejecuta las celdas en orden con **Kernel → Restart & Run All** (o **Ejecutar todo**). Conservaremos el DataFrame original para comparar los datos antes y después de la limpieza.


## Preparación del entorno

Se utilizará **Pandas** para leer, inspeccionar, limpiar y exportar los datos. Si no tienes instalada la librería, ejecuta en una terminal: `python -m pip install pandas`.

El notebook genera `pokemon_limpio.csv` en la misma carpeta donde se encuentra.


In [ ]:
from pathlib import Path
import pandas as pd

# Ruta del archivo fuente. Debe estar junto a este notebook.
archivo_entrada = Path("pokemon.csv")
archivo_salida = Path("pokemon_limpio.csv")

if not archivo_entrada.exists():
    raise FileNotFoundError(
        "No se encontró pokemon.csv. Coloca el CSV en la misma carpeta que este notebook."
    )

print("Pandas:", pd.__version__)
print("Archivo de entrada:", archivo_entrada.resolve())


# Fase 1. Extract — Extracción e inspección inicial

En esta fase se carga el CSV, se conserva una copia sin modificar y se revisan la estructura, las dimensiones, los tipos de datos, las estadísticas y los valores nulos.


In [ ]:
# Leer el archivo original y crear una copia de trabajo
dataFrame_Original = pd.read_csv(archivo_entrada)
dataFrame_Limpio = dataFrame_Original.copy()

print("Primeras 5 filas del conjunto original:")
display(dataFrame_Original.head())


In [ ]:
print("Cantidad de filas y columnas:", dataFrame_Original.shape)
print("\nNombres de las columnas:")
print(dataFrame_Original.columns.tolist())

print("\nEstructura y tipos de datos:")
dataFrame_Original.info()


In [ ]:
print("Estadísticas descriptivas de los datos originales:")
display(dataFrame_Original.describe(include="all").T)


In [ ]:
print("Valores nulos por columna en el conjunto original:")
display(dataFrame_Original.isnull().sum().to_frame("Valores nulos"))


### Observaciones iniciales

Al inspeccionar el archivo pueden detectarse, entre otros problemas:

- Espacios al inicio o al final de algunos textos.
- Filas con campos desplazados, como el registro de Beedrill y varios Pokémon monotipo.
- Errores tipográficos o variantes en los nombres de los tipos.
- Valores nulos que pueden representar un segundo tipo ausente o un dato perdido por el desplazamiento.
- Columnas numéricas guardadas como texto y valores que requieren conversión de tipo.

Las siguientes celdas aplican correcciones explícitas a los casos descritos en las instrucciones de la práctica y observados en este archivo.


# Fase 2. Transform — Transformación y limpieza

## 2.1 Eliminar espacios en blanco

`str.strip()` quita espacios al inicio y al final de las columnas de texto. Esto evita que valores iguales sean tratados como categorías diferentes.


In [ ]:
columnas_texto = dataFrame_Limpio.select_dtypes(include=["object"]).columns

for columna in columnas_texto:
    dataFrame_Limpio[columna] = dataFrame_Limpio[columna].str.strip()

print("Espacios externos eliminados de las columnas de texto.")
display(dataFrame_Limpio.head())


## 2.2 Reorganizar registros desplazados

### Caso Beedrill
En el archivo original, el texto `15 Beedrill` aparece en `N.º` y los demás valores se desplazan una columna. Se restaura la fila con los campos correspondientes.

### Pokémon monotipo
Cuando `Tipo2` contiene un número en lugar de un tipo, se interpreta como un desplazamiento: ese número corresponde a la altura, el valor de `Altura` corresponde al peso y el peso desplazado aparece como `0.0`. Para los Pokémon indicados por la práctica, se restauran los campos y se asigna `Ninguno` a `Tipo2`.

**Nota:** estas correcciones son reglas específicas para los registros identificados; en un proyecto real, se validarían contra una fuente de referencia confiable.


In [ ]:
# Corregir el registro desplazado de Beedrill
fila_beedrill = dataFrame_Limpio["N.º"].astype(str).str.contains("Beedrill", na=False)

dataFrame_Limpio.loc[
    fila_beedrill,
    ["N.º", "Nombre", "Tipo1", "Tipo2", "Altura", "Peso", "Legendario"]
] = ["15", "Beedrill", "Bicho", "Veneno", 1.0, 29.5, 0]

# Restaurar los Pokémon monotipo identificados en las instrucciones
correcciones_monotipo = {
    "Rattata":   (0.3, 3.5, 0),
    "Raticate":  (0.7, 18.5, 0),
    "Dratini":   (1.8, 3.3, 0),
    "Dragonair": (4.0, 16.5, 0),
    "Mewtwo":    (2.0, 122.0, 1),
}

for nombre, (altura_correcta, peso_correcto, legendario_correcto) in correcciones_monotipo.items():
    fila = dataFrame_Limpio["Nombre"].eq(nombre)
    dataFrame_Limpio.loc[fila, "Tipo2"] = "Ninguno"
    dataFrame_Limpio.loc[fila, "Altura"] = altura_correcta
    dataFrame_Limpio.loc[fila, "Peso"] = peso_correcto
    dataFrame_Limpio.loc[fila, "Legendario"] = legendario_correcto

# Corregir el identificador de Articuno
dataFrame_Limpio["N.º"] = dataFrame_Limpio["N.º"].replace({"14 4": "144"})

# Mostrar los registros corregidos para revisar visualmente
nombres_revisar = ["Beedrill", "Rattata", "Raticate", "Dratini", "Dragonair", "Mewtwo", "Articuno"]
display(dataFrame_Limpio[dataFrame_Limpio["Nombre"].isin(nombres_revisar)])


## 2.3 Estandarizar categorías y corregir errores tipográficos

Se unifican sinónimos, palabras mal escritas y variantes de mayúsculas/minúsculas para que una misma categoría tenga una representación consistente.


In [ ]:
correcciones_tipos = {
    "Wa ter": "Agua",
    "Hierba": "Planta",
    "Insecto": "Bicho",
    "I ce": "Hielo",
    "Veneno n": "Veneno",
    "luchando": "Lucha",
    "Lucha ng": "Lucha",
    "psíquico": "Psíquico",
    "hada": "Hada",
}

for columna in ["Tipo1", "Tipo2"]:
    dataFrame_Limpio[columna] = dataFrame_Limpio[columna].replace(correcciones_tipos)

# Corrección adicional del nombre mal escrito presente en el CSV
dataFrame_Limpio["Nombre"] = dataFrame_Limpio["Nombre"].replace(
    {"Bellspr fuera": "Bellsprout"}
)

# Inspeccionar categorías después de la normalización
print("Categorías únicas en Tipo1:")
print(sorted(dataFrame_Limpio["Tipo1"].dropna().unique()))

print("\nCategorías únicas en Tipo2 antes de rellenar nulos:")
print(sorted(dataFrame_Limpio["Tipo2"].dropna().unique()))


## 2.4 Tratar valores faltantes

- En `Tipo2`, un valor nulo puede representar un Pokémon monotipo; se reemplaza por la categoría `Ninguno`.
- En `Legendario`, los valores nulos restantes se reemplazan por `0`, como solicita la práctica. Esta regla supone que los nulos son Pokémon no legendarios; en un análisis real, convendría verificar cada caso antes de asumirlo.


In [ ]:
dataFrame_Limpio["Tipo2"] = dataFrame_Limpio["Tipo2"].fillna("Ninguno")
dataFrame_Limpio["Legendario"] = dataFrame_Limpio["Legendario"].fillna(0)

print("Valores nulos después del tratamiento:")
display(dataFrame_Limpio.isnull().sum().to_frame("Valores nulos"))


## 2.5 Corregir tipos de datos

`N.º` debe ser un entero; `Altura` y `Peso` deben ser numéricos; y `Legendario` se convierte a booleano (`True` o `False`).


In [ ]:
dataFrame_Limpio["N.º"] = pd.to_numeric(dataFrame_Limpio["N.º"], errors="raise").astype(int)
dataFrame_Limpio["Altura"] = pd.to_numeric(dataFrame_Limpio["Altura"], errors="raise")
dataFrame_Limpio["Peso"] = pd.to_numeric(dataFrame_Limpio["Peso"], errors="raise")
dataFrame_Limpio["Legendario"] = dataFrame_Limpio["Legendario"].astype(bool)

# Usar N.º como índice para identificar registros y devolverlo después como columna
dataFrame_Limpio = dataFrame_Limpio.set_index("N.º")
dataFrame_Limpio = dataFrame_Limpio.reset_index()

print("Tipos de datos después de la conversión:")
display(dataFrame_Limpio.dtypes.to_frame("Tipo de dato"))


# Fase 3. Load — Exportación del conjunto limpio

Se exporta el DataFrame a `pokemon_limpio.csv`. El argumento `index=False` evita guardar un índice adicional como columna; `utf-8-sig` facilita la apertura del archivo en programas como Excel.


In [ ]:
dataFrame_Limpio.to_csv(archivo_salida, index=False, encoding="utf-8-sig")

print("Archivo exportado correctamente:", archivo_salida.resolve())
print("Tamaño del conjunto final (filas, columnas):", dataFrame_Limpio.shape)


# Verificación del resultado

Se comprueba la integridad del DataFrame final mediante las revisiones solicitadas en la práctica.


In [ ]:
print("Primeras 10 filas:")
display(dataFrame_Limpio.head(10))


In [ ]:
print("Últimas 10 filas:")
display(dataFrame_Limpio.tail(10))


In [ ]:
print("10 registros aleatorios:")
display(dataFrame_Limpio.sample(n=min(10, len(dataFrame_Limpio)), random_state=42))


In [ ]:
print("Estructura del DataFrame final:")
dataFrame_Limpio.info()


In [ ]:
print("Estadísticas descriptivas del DataFrame final:")
display(dataFrame_Limpio.describe(include="all").T)


In [ ]:
print("Valores nulos por columna:")
display(dataFrame_Limpio.isnull().sum().to_frame("Valores nulos"))

print("\nDimensiones finales:", dataFrame_Limpio.shape)
print("Total de registros:", len(dataFrame_Limpio))


In [ ]:
print("Tipos de Pokémon sin repetir (Tipo1):")
tipos_unicos = sorted(dataFrame_Limpio["Tipo1"].dropna().unique())
for tipo in tipos_unicos:
    print("-", tipo)

print("\nCantidad de categorías únicas:", len(tipos_unicos))


## Comprobaciones adicionales de calidad

Además de los puntos solicitados, se revisa que los identificadores no estén duplicados y que las columnas numéricas no contengan valores negativos. Estas comprobaciones no sustituyen la validación contra una fuente oficial, pero ayudan a detectar anomalías.


In [ ]:
print("IDs duplicados:", dataFrame_Limpio["N.º"].duplicated().sum())
print("Alturas negativas:", (dataFrame_Limpio["Altura"] < 0).sum())
print("Pesos negativos:", (dataFrame_Limpio["Peso"] < 0).sum())
print("Valores distintos de Legendario:", sorted(dataFrame_Limpio["Legendario"].unique().tolist()))

# Confirmar que el CSV exportado puede leerse de nuevo
verificacion_csv = pd.read_csv(archivo_salida)
print("\nCSV exportado: dimensiones al volver a leerlo =", verificacion_csv.shape)


# Pregunta de análisis

## ¿Qué cambios faltan por realizar en el set de datos resultante y cómo los aplicarías?

**Respuesta sugerida para revisar y adaptar:**

Aunque se corrigieron los errores identificados en la práctica, todavía sería conveniente validar los nombres, tipos, alturas, pesos e identificadores contra una fuente de referencia confiable. También revisaría valores atípicos, duplicados y reglas de consistencia entre las columnas. Para hacerlo, utilizaría filtros de Pandas, `duplicated()` para buscar registros repetidos, comparaciones con un catálogo oficial y reglas de validación para comprobar rangos permitidos. No modificaría datos dudosos sin verificar primero su valor correcto.

# Conclusiones

**Borrador para personalizar:**

En esta práctica se implementó un proceso ETL con Pandas para leer, inspeccionar, limpiar y exportar un conjunto de datos. Durante la transformación se normalizaron espacios y categorías, se corrigieron registros desplazados, se trataron valores faltantes y se convirtieron columnas a los tipos de datos adecuados. Finalmente, se exportó el archivo `pokemon_limpio.csv` y se verificaron sus dimensiones, estructura, estadísticas, valores nulos y categorías de Pokémon. La actividad permitió comprender la importancia de revisar la calidad de los datos antes de utilizarlos para análisis.

# Referencias

- Documento de instrucciones de la práctica: *Práctica 2. Pipeline ETL y Limpieza de Datos con Pandas*.
- Documentación oficial de Pandas: https://pandas.pydata.org/docs/

> **Antes de entregar:** escribe tu nombre y fecha, ejecuta todas las celdas en orden, revisa los resultados y toma capturas de pantalla como evidencia. Comprueba que el archivo `pokemon_limpio.csv` se haya generado en la carpeta del notebook.
